# 02 · Chunking
Step 2: build the fixed-size and semantic chunk sets (tables shared by both), inspect them, save to `index/chunks/`.

Sections 1–4 need no API key. Section 5 (semantic) calls the OpenAI embeddings API.

After editing anything in `src/`, **restart the kernel** (execution counts should reset to 1).

In [1]:
import sys, pathlib
print(sys.executable)  # should end with finsight-ai\.venv\Scripts\python.exe
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import Markdown, display
from finsight.config import list_documents
from finsight.ingest.parsers import parse_document
from finsight.chunking.pipeline import split_elements, build_and_save, chunks_path
from finsight.chunking.base import is_heading

c:\Users\senpr\Documents\finsight-ai\.venv\Scripts\python.exe


## 1. Parse once
Reused by every chunker below (~30 s).

In [2]:
docs = list_documents()
parsed = {d.doc_id: parse_document(d) for d in docs}
{k: len(v) for k, v in parsed.items()}

{'alphabet_2023': 209,
 'alphabet_2024': 217,
 'alphabet_2025': 219,
 'amazon_2023': 163,
 'amazon_2024': 158,
 'amazon_2025': 159,
 'microsoft_2023': 136,
 'microsoft_2024': 140,
 'microsoft_2025': 126}

## 2. Check section detection
Headings found per document. Look for real section names (`Item 7. ...`, `Segment Information`), not noise.

In [3]:
for d in docs:
    blocks, tables = split_elements(parsed[d.doc_id])
    sections = list(dict.fromkeys(b.section for b in blocks))
    print(f'{d.doc_id}: {len(blocks)} blocks, {len(tables)} tables, {len(sections)} sections')
    print('   ', sections[:15])

alphabet_2023: 101 blocks, 56 tables, 44 sections
    ['Securities And Exchange Commission', 'Form 10-K', 'Documents Incorporated By Reference', 'Table Of Contents', 'Part I', 'Part II', 'Part III', 'Part IV', 'Item 1. Business', 'Item 1A. Risk Factors', 'Item 1B. Unresolved Staff Comments', 'Item 1C. Cybersecurity', 'Item 2. Properties', 'Item 3. Legal Proceedings', 'Item 4. Mine Safety Disclosures']
alphabet_2024: 104 blocks, 59 tables, 44 sections
    ['Securities And Exchange Commission', 'Form 10-K', 'Documents Incorporated By Reference', 'Table Of Contents', 'Part I', 'Part II', 'Part III', 'Part IV', 'Item 1. Business', 'Item 1A. Risk Factors', 'Item 1B. Unresolved Staff Comments', 'Item 1C. Cybersecurity', 'Item 2. Properties', 'Item 3. Legal Proceedings', 'Item 4. Mine Safety Disclosures']
alphabet_2025: 105 blocks, 60 tables, 44 sections
    ['Securities And Exchange Commission', 'Form 10-K', 'Documents Incorporated By Reference', 'Table Of Contents', 'Part I', 'Part II', 'Pa

## 3. Fixed-size chunks (512 tokens, 64 overlap) + table chunks
No API calls.

In [4]:
fixed = build_and_save('fixed', parsed)
df = pd.DataFrame([{'doc_id': c.doc_id, 'kind': c.kind, 'section': c.section, 'page': c.page, 'tokens': c.n_tokens} for c in fixed])
print('saved to', chunks_path('fixed'))
df.groupby(['doc_id', 'kind']).tokens.agg(['count', 'median', 'max']).unstack()

saved to C:\Users\senpr\Documents\finsight-ai\index\chunks\fixed.jsonl


count      median          max     
kind           table text  table   text table text
doc_id                                            
alphabet_2023     58  189  196.5  433.0   801  531
alphabet_2024     61  195  199.0  417.0   779  532
alphabet_2025     63  193  195.0  434.0   804  535
amazon_2023       37  178  261.0  422.0   798  540
amazon_2024       35  173  272.0  421.0   797  540
amazon_2025       35  177  289.0  428.0   795  541
microsoft_2023    69  149  201.0  346.0   788  522
microsoft_2024    72  146  190.0  313.5   796  527
microsoft_2025    68  129  219.5  277.0   799  520

In [5]:
df.groupby('kind').tokens.describe()

,count,mean,std,min,25%,50%,75%,max
kind,,,,,,,,
table,498.0,271.321285,174.386920,84.0,148.0,215.0,322.75,804.0
text,1529.0,347.499019,160.377015,32.0,200.0,413.0,488.00,541.0


### Sample text chunk

In [6]:
print(next(c for c in fixed if c.kind == 'text' and c.doc_id == 'amazon_2024' and c.n_tokens > 300).text)

Amazon | FY2024 | annual_report | General | p.2
Apart from the financial results, we made our customers’ lives meaningfully better and easier. In our Stores business, we substantially expanded selection, continued lowering prices (independent research firm, Profitero, found Amazon the lowest-priced online U.S. Retailer for the eighth year in a row), and for the second year in a row, we shipped at record speed to our Prime members. AWS launched a slew of new infrastructure and AI services that make it even easier to build remarkable customer experiences, including our latest custom AI silicon (Trainium2), a new set of frontier foundation models in Amazon Nova, and significant expansion of available models and features in our leading Generative AI (“GenAI”) services Amazon SageMaker and Amazon Bedrock. Prime Video continued to offer compelling original shows, including new seasons for Fallout, Reacher, The Boys, and The Lord of the Rings: Rings of Power, movies like Road House, The Idea 

## 4. Table chunks
Check title, units and header. `embed_text` is what gets embedded; `text` is what the LLM sees.

In [7]:
tables = [c for c in fixed if c.kind == 'table']
def show_table(doc_id, i):
    t = [c for c in tables if c.doc_id == doc_id][i]
    print(t.chunk_id, '| title:', t.extra['title'], '| units:', t.extra['units'] or '-', '| groups:', t.extra['n_groups'])
    print('EMBED:', t.embed_text[:400])
    display(Markdown(t.text[:2500]))
show_table('amazon_2024', 10)

amazon_2024-table-0009-g1 | title: CONSOLIDATED STATEMENTS OF CASH FLOWS | units: in millions | groups: 2
EMBED: Amazon annual_report FY2024. Table: CONSOLIDATED STATEMENTS OF CASH FLOWS. Section: Consolidated Statements Of Cash Flows. Units: in millions. Columns: 2022, 2023, 2024. Rows: Principal repayments of financing obligations; Net cash provided by (used in) financing activities; Foreign currency effect on cash, cash equivalents, and restricted cash; Net increase (decrease) in cash, cash equivalents, a


Amazon | FY2024 | annual_report | Consolidated Statements Of Cash Flows | p.48
Table: CONSOLIDATED STATEMENTS OF CASH FLOWS (rows 31-35 of 35)
Units: in millions
|  | 2022 | 2023 | 2024 |
|---|---|---|---|
| Principal repayments of financing obligations | (248) | (271) | (669) |
| Net cash provided by (used in) financing activities | 9,718 | (15,879) | (11,812) |
| Foreign currency effect on cash, cash equivalents, and restricted cash | (1,093) | 403 | (1,301) |
| Net increase (decrease) in cash, cash equivalents, and restricted cash | 17,776 | 19,637 | 8,422 |
| CASH, CASH EQUIVALENTS, AND RESTRICTED CASH, END OF PERIOD | $54,253 | $73,890 | $82,312 |

In [8]:
show_table('alphabet_2024', 10)

alphabet_2024-table-0010 | title: The following table presents segment operating income (loss) (in millions). | units: in millions | groups: 1
EMBED: Alphabet 10-K FY2024. Table: The following table presents segment operating income (loss) (in millions).. Section: Operations. Units: in millions. Rows: Google Services; Google Cloud; Other Bets; Alphabet-level activities; Total income from operations.


Alphabet | FY2024 | 10-K | Operations | p.40
Table: The following table presents segment operating income (loss) (in millions).
Units: in millions
| Operating income (loss): |  |  |  |
|---|---|---|---|
| Google Services |  | $95,858 | $121,263 |
| Google Cloud |  | 1,716 | 6,112 |
| Other Bets |  | (4,095) | (4,444) |
| Alphabet-level activities | (1) | (9,186) | (10,541) |
| Total income from operations |  | $84,293 | $112,390 |

In [9]:
show_table('microsoft_2024', 4)

microsoft_2024-table-0004 | title: SEGMENT RESULTS OF OPERATIONS | units: in millions, except percentages | groups: 1
EMBED: Microsoft annual_report FY2024. Table: SEGMENT RESULTS OF OPERATIONS. Section: Segment Results Of Operations. Units: in millions, except percentages. Columns: 2024, 2023, Percentage Change. Rows: Revenue; Productivity and Business Processes; Intelligent Cloud; More Personal Computing; Total; Operating Income; Productivity and Business Processes; Intelligent Cloud; More Personal Computing; Total.


Microsoft | FY2024 | annual_report | Segment Results Of Operations
Table: SEGMENT RESULTS OF OPERATIONS
Units: in millions, except percentages
| (In millions, except percentages) | 2024 | 2023 | Percentage Change |
|---|---|---|---|
| Revenue |  |  |  |
| Productivity and Business Processes | $ 77,728 | $ 69,274 | 12% |
| Intelligent Cloud | 105,362 | 87,907 | 20% |
| More Personal Computing | 62,032 | 54,734 | 13% |
| Total | $ 245,122 | $ 211,915 | 16% |
| Operating Income |  |  |  |
| Productivity and Business Processes | $ 40,540 | $ 34,189 | 19% |
| Intelligent Cloud | 49,584 | 37,884 | 31% |
| More Personal Computing | 19,309 | 16,450 | 17% |
| Total | $ 109,433 | $ 88,523 | 24% |

### Title / units coverage
Share of tables with a detected title (not just the section name) and units.

In [10]:
tdf = pd.DataFrame([{'doc_id': c.doc_id, 'has_units': bool(c.extra['units']), 'title_is_section': c.extra['title'] == c.section,
                     'has_header': bool(c.extra['columns']), 'split': c.extra['n_groups'] > 1} for c in tables])
tdf.groupby('doc_id').mean().round(2)

,has_units,title_is_section,has_header,split
doc_id,,,,
alphabet_2023,0.64,0.00,0.71,0.07
alphabet_2024,0.67,0.00,0.82,0.07
alphabet_2025,0.62,0.02,0.65,0.10
amazon_2023,0.68,0.00,0.59,0.05
amazon_2024,0.69,0.00,0.60,0.06
amazon_2025,0.71,0.00,0.66,0.06
microsoft_2023,0.94,0.01,1.00,0.03
microsoft_2024,0.94,0.01,1.00,0.06
microsoft_2025,0.93,0.03,1.00,0.15


## 5. Semantic chunks (calls the OpenAI API)
Embeds every sentence with `text-embedding-3-small` (cached in `index/embeddings_cache/`). Roughly 600K tokens for the whole corpus, a few cents. Needs `OPENAI_API_KEY` in `.env`.

Try one document first.

In [11]:
from finsight.chunking.semantic import chunk_text_semantic
d = next(d for d in docs if d.doc_id == 'amazon_2024')
blocks, _ = split_elements(parsed[d.doc_id])
sem_one = chunk_text_semantic(d, blocks)
pd.Series([c.n_tokens for c in sem_one]).describe()

count    154.000000
mean     408.097403
std      240.103838
min       28.000000
25%      227.250000
50%      365.500000
75%      597.000000
max      832.000000
dtype: float64

In [12]:
print(sem_one[5].text)

Amazon | FY2024 | annual_report | General | p.4
It led to some people overly digging in and fighting for their ideas. There’s nothing wrong with pushing for what you believe. But, in my experience, the best leaders want to hear others’ views. They don’t wilt or bristle when challenged; they’re intrigued. Effective leaders change their minds when presented with new compelling information (which makes it ironic how people dismiss politicians as “flip-floppers” when they change their position). Ultimately, leaders are responsible for getting to the best answer for customers, regardless of whose original idea is chosen. Learn and be Curious “Leaders are never done learning and always seek to improve themselves. They are curious about new possibilities and act to explore them.” In the nearly 28 years I’ve been at Amazon, the biggest difference in the relative growth of companies and individuals has been their aptitude to learn. At a certain point, some leaders seem to lose their thirst to l

When the single document looks right, build and save the full semantic set:

In [13]:
semantic = build_and_save('semantic', parsed)
sdf = pd.DataFrame([{'doc_id': c.doc_id, 'kind': c.kind, 'tokens': c.n_tokens} for c in semantic])
print('saved to', chunks_path('semantic'))
sdf.groupby(['doc_id', 'kind']).tokens.agg(['count', 'median', 'max']).unstack()

saved to C:\Users\senpr\Documents\finsight-ai\index\chunks\semantic.jsonl


count      median          max     
kind           table text  table   text table text
doc_id                                            
alphabet_2023     58  167  196.5  339.0   801  822
alphabet_2024     61  177  199.0  304.0   779  822
alphabet_2025     63  177  195.0  299.0   804  823
amazon_2023       37  159  261.0  351.0   798  835
amazon_2024       35  154  272.0  365.5   797  832
amazon_2025       35  152  289.0  383.0   795  819
microsoft_2023    69  136  201.0  314.0   788  804
microsoft_2024    72  139  190.0  308.0   796  810
microsoft_2025    68  126  219.5  268.0   799  810

## 6. Compare the two sets

In [14]:
pd.DataFrame({
    'fixed': df[df.kind == 'text'].tokens.describe(),
    'semantic': sdf[sdf.kind == 'text'].tokens.describe(),
}).round(0)

,fixed,semantic
count,1529.0,1387.0
mean,347.0,369.0
std,160.0,237.0
min,32.0,28.0
25%,200.0,179.0
50%,413.0,326.0
75%,488.0,538.0
max,541.0,835.0


## Notes / issues found
_(fill in after reviewing)_